# W&B Confusion Matrix Provenance Notebook

- Purpose: reconstruct the supplemental ViT confusion matrix (Fig S3) from the bundled W&B table artifact
- Checkpoint: `smqmqi14` (confusion matrix artifact from W&B run `76zuzb4h`)
- Artifact: `conf_mat_top1_inference_xrd_model_smqmqi14_table_2_6491e754dbab738e58a0.table.json` — bundled at `assets/figure_data/`

**Dependency:** `pip install seaborn` (not in base `environment-train-eval.yml`)

This notebook does not require a GPU or large HDF5 datasets — it renders directly from the stored W&B table JSON.

In [ ]:
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None
)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

file_path = str(
    REPO_ROOT / 'assets' / 'figure_data' /
    'conf_mat_top1_inference_xrd_model_smqmqi14_table_2_6491e754dbab738e58a0.table.json'
)
print(f'Artifact: {file_path}')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import json

# Load JSON file
with open(file_path, "r") as f:
    data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(data["data"], columns=data["columns"])

num_classes = 99
conf_matrix = np.zeros((num_classes, num_classes), dtype=float)
for _, row in df.iterrows():
    actual = int(row["Actual"])
    predicted = int(row["Predicted"])
    weight = float(row["nPredictions"])
    conf_matrix[actual, predicted] += weight

plt.figure(figsize=(16, 12))
sns.heatmap(conf_matrix, cmap="Blues", annot=False)
plt.title("ViT Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig("confusion_matrix.png")
plt.show()

In [ ]:
# Normalize per row
row_sums = conf_matrix.sum(axis=1, keepdims=True)
conf_matrix_norm = np.divide(
    conf_matrix, row_sums,
    out=np.zeros_like(conf_matrix),
    where=row_sums != 0
)
plt.figure(figsize=(20, 16))
sns.heatmap(conf_matrix_norm, cmap="viridis", annot=False, cbar=True)
plt.title("ViT Confusion Matrix (Row-normalized)")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig("confusion_matrix_normalized.png")
plt.show()